# **🟢 CELL 1: Install Dependencies**

In [1]:
!pip install -q fastapi uvicorn pyngrok nest_asyncio google-genai pydantic requests transformers peft bitsandbytes accelerate

# **🟢 CELL 2: Imports and Configurations**

In [ ]:
import os
import shutil
import mimetypes
import tempfile
import threading
import requests
from contextlib import asynccontextmanager
from typing import List, Optional

import nest_asyncio
import torch
import uvicorn
from fastapi import FastAPI, UploadFile, File, Form, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel
from pyngrok import ngrok
from google import genai
from google.genai import types
from kaggle_secrets import UserSecretsClient

# define user secrets
user_secrets = UserSecretsClient()

# Patch asyncio loop for Jupyter notebook environment
nest_asyncio.apply()

# Gemini Vision & API Configuration
api_keys = [user_secrets.get_secret("GEMINI_API_KEY")]
MODEL_FALLBACKS = ["gemini-2.0-flash","gemini-2.5-flash", "gemini-1.5-flash"]


In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

MODEL_ID = "Mazenbassem/fiTpulse-fitness-model"
BASE_MODEL = "unsloth/Qwen2.5-7B-Instruct-bnb-4bit"

print("🚀 Loading base model into Kaggle GPU...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    device_map="auto"
)

print("⚡ Applying your fine-tuned FitPulse weights...")
model = PeftModel.from_pretrained(base_model, MODEL_ID)

print("✅ FitPulse AI Model Ready!")

🚀 Loading base model into Kaggle GPU...


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

⚡ Applying your fine-tuned FitPulse weights...
✅ FitPulse AI Model Ready!


# **🟢 CELL 3: Helper Funtions(***Rag + Parser + Image Parser***)**

In [ ]:
def parse_inbody_image(file_path: str) -> str:
    if not os.path.exists(file_path):
        return "Failed to parse file: File not found."

    with open(file_path, "rb") as f:
        file_bytes = f.read()

    # Dynamic MIME detection (PNG, JPEG, PDF)
    mime_type, _ = mimetypes.guess_type(file_path)
    if not mime_type:
        ext = os.path.splitext(file_path)[1].lower()
        if ext == ".pdf":
            mime_type = "application/pdf"
        elif ext == ".png":
            mime_type = "image/png"
        else:
            mime_type = "image/jpeg"

    prompt_text = """Analyze this scan/document sheet and extract key metrics:
1. Weight (kg)
2. Skeletal Muscle Mass / SMM (kg)
3. Body Fat Mass / BFM (kg)
4. Percent Body Fat / PBF (%)
5. Basal Metabolic Rate / BMR (kcal)
6. Muscle-Fat Graph Shape (C-shape, I-shape, or D-shape)
7. Segmental Lean Analysis (% of Standard)
8. Extra relevant details.

Return ONLY a concise bulleted list."""

    for key_idx, key in enumerate(api_keys, start=1):
        client = genai.Client(api_key=key)
        for model_name in MODEL_FALLBACKS:
            try:
                response = client.models.generate_content(
                    model=model_name,
                    contents=[
                        types.Part.from_bytes(data=file_bytes, mime_type=mime_type),
                        prompt_text
                    ]
                )
                if response.text and response.text.strip():
                    return response.text.strip()
            except Exception as e:
                print(f"⚠️ Error with model '{model_name}': {e}")  # Logs the real error in Kaggle output
                continue

    return "Failed to parse file after exhausting API options."


def process_inbody_and_consult(file_path: str, user_goal: str = "Recomposition") -> str:
    inbody_summary = parse_inbody_image(file_path)
    
    combined_prompt = f"""Here is my scan/report summary:
{inbody_summary}

User Goal: {user_goal}

Based on these numbers and my goal, provide:
1. Analysis of muscle-to-fat balance.
2. Tailored workout strategy.
3. Daily calories and protein targets based on BMR."""

    # return ask_fitpulse_with_memory(
    #     prompt=combined_prompt,
    #     use_rag=True
    # )
    return stream_fitpulse_with_memory(
        prompt=combined_prompt,
        use_rag=True
    )

import json
def parse_nutrition_label(file_path: str) -> str:
    if not os.path.exists(file_path):
        return "Failed to parse file: File not found."

    with open(file_path, "rb") as f:
        file_bytes = f.read()

    # Dynamic MIME detection (PNG, JPEG, PDF)
    mime_type, _ = mimetypes.guess_type(file_path)
    if not mime_type:
        ext = os.path.splitext(file_path)[1].lower()
        if ext == ".pdf":
            mime_type = "application/pdf"
        elif ext == ".png":
            mime_type = "image/png"
        else:
            mime_type = "image/jpeg"

    prompt_text = """Analyze this nutrition facts label image and extract key metrics PER SERVING:
1. Serving Size
2. Servings Per Container
3. Calories (kcal)
4. Total Fat (g)
5. Sodium (mg)
6. Total Carbohydrates (g)
7. Dietary Fiber (g)
8. Total Sugars (g)
9. Protein (g)
10. Extra relevant nutrition details.

Return ONLY a concise bulleted list."""

    for key_idx, key in enumerate(api_keys, start=1):
        client = genai.Client(api_key=key)
        for model_name in MODEL_FALLBACKS:
            try:
                response = client.models.generate_content(
                    model=model_name,
                    contents=[
                        types.Part.from_bytes(data=file_bytes, mime_type=mime_type),
                        prompt_text
                    ]
                )
                if response.text and response.text.strip():
                    return response.text.strip()
            except Exception as e:
                #print(e)
                continue

    return "Failed to parse file after exhausting API options."

In [ ]:
from threading import Thread
from transformers import TextIteratorStreamer
import requests

FITPULSE_SYSTEM_PROMPT = (
    "You are FitPulse AI, an expert fitness, nutrition, and body composition coach. "
    "Format every reply in Markdown: use ## / ### headings to organize sections, "
    "**bold** for key numbers/terms, and - bullet or 1. numbered lists for steps, make sure h markdown elements can be rendered. "
    "Whenever you recommend a specific named exercise (not a general category), "
    "immediately follow its name with a tag on its own line in this exact format: "
    "[EXERCISE: {\"id\": \"lowercase_snake_case_id\", \"name\": \"Human Readable Exercise Name\"}] "
    "so the app can display a reference photo. Only tag exercises you are actively "
    "recommending, not passing mentions."
)

def stream_fitpulse_with_memory(
    prompt: str,
    history: Optional[List[dict]] = None,
    system_prompt: str = FITPULSE_SYSTEM_PROMPT,
    use_rag: bool = False
):
    """Streams token chunks directly from Kaggle GPU memory using PyTorch."""
    messages = [{"role": "system", "content": system_prompt}]
    if history:
        for msg in history:
            messages.append({"role": msg["role"], "content": msg["content"]})

    context_str = ""
    if use_rag and 'index' in globals():
        try:
            retriever = index.as_retriever(similarity_top_k=2)
            nodes = retriever.retrieve(prompt)
            if nodes:
                retrieved_chunks = [f"- {node.get_content()}" for node in nodes]
                context_str = "\n\nReference Knowledge:\n" + "\n".join(retrieved_chunks)
        except Exception as e:
            print(f"⚠️ RAG Retrieval warning: {e}")

    messages.append({"role": "user", "content": f"{prompt}{context_str}"})

    # Detect device dynamically (CUDA GPU or CPU fallback)
    device = "cuda" if torch.cuda.is_available() else "cpu"

    # 1. Format prompt with return_dict=True to get tensors for input_ids and attention_mask
    model_inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=True
    ).to(device)

    # 2. Set up real-time streaming streamer
    streamer = TextIteratorStreamer(tokenizer, skip_prompt=True, skip_special_tokens=True)
    
    # 3. Unpack model_inputs into generation_kwargs
    generation_kwargs = dict(
        **model_inputs,
        streamer=streamer,
        max_new_tokens=1024,
        temperature=0.7,
        top_p=0.9,
        do_sample=True,
    )

    thread = Thread(target=model.generate, kwargs=generation_kwargs)
    thread.start()
    
    # 4. Stream tokens using Dynamic Queue Draining for maximum network speed
    import queue

    while True:
        try:
            # Wait for the next token from the generator thread
            first_text = streamer.text_queue.get(timeout=15.0)
            if first_text is streamer.stop_signal:
                break

            chunk = first_text

            # Drain any extra tokens that accumulated in the queue while sending HTTP bytes
            while not streamer.text_queue.empty():
                next_text = streamer.text_queue.get_nowait()
                if next_text is streamer.stop_signal:
                    yield chunk
                    chunk = ""
                    break
                chunk += next_text

            if chunk:
                yield chunk

        except queue.Empty:
            break
   

# **🟢 CELL 4: FASTAPI**

In [ ]:
import base64
import tempfile
import os
from typing import List, Optional
from contextlib import asynccontextmanager

from fastapi import FastAPI, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from fastapi.responses import StreamingResponse
from pydantic import BaseModel

# Safe handling of optional modules in lifespan
try:
    from pyngrok import ngrok
except ImportError:
    ngrok = None

try:
    import torch
except ImportError:
    torch = None


@asynccontextmanager
async def lifespan(app: FastAPI):
    print("🚀 FitPulse API starting up...")
    yield
    print("🛑 Shutting down FitPulse API...")
    if ngrok:
        try:
            ngrok.kill()
        except Exception:
            pass
    if torch and torch.cuda.is_available():
        torch.cuda.empty_cache()


app = FastAPI(title="FitPulse AI Backend API", lifespan=lifespan)

# Allow cross-origin requests from frontend / ngrok dashboard
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=False,
    allow_methods=["*"],
    allow_headers=["*"],
    expose_headers=["*"],
)

class ChatMessage(BaseModel):
    role: str
    content: str

class ChatRequest(BaseModel):
    query: str
    inbody_image: Optional[str] = None
    label_image: Optional[str] = None  # Added
    history: Optional[List[ChatMessage]] = []
    use_rag: Optional[bool] = True


@app.get("/")
def health_check():
    return {"status": "online", "model": "FitPulse AI (Kaggle Backend)"}


def to_stream_generator(result):
    """
    Flexible streamer: Yields tokens immediately if your notebook function returns a 
    generator/streamer, or safely chunks the output if it returns a single string.
    """
    if isinstance(result, str):
        chunk_size = 32
        for i in range(0, len(result), chunk_size):
            yield result[i : i + chunk_size]
    else:
        # Handles generators, TextIteratorStreamer, or chunk lists
        for chunk in result:
            if isinstance(chunk, str):
                yield chunk
            elif hasattr(chunk, "text"):
                yield chunk.text
            elif isinstance(chunk, dict) and "content" in chunk:
                yield chunk["content"]


STREAM_HEADERS = {
    "X-Accel-Buffering": "no",
    "Cache-Control": "no-cache",
    "Connection": "keep-alive",
}

@app.post("/api/chat")
async def chat_endpoint(req: ChatRequest):
    # 🔍 DEBUG LOG: Check incoming payload state
    print("\n--- [BACKEND DEBUG] New Chat Request Received ---")
    print(f"Query: '{req.query}'")
    print(f"inbody_image received: {bool(req.inbody_image and req.inbody_image.strip())} (len: {len(req.inbody_image) if req.inbody_image else 0})")
    print(f"label_image received:  {bool(req.label_image and req.label_image.strip())} (len: {len(req.label_image) if req.label_image else 0})")
    
    try:
        formatted_history = (
            [{"role": msg.role, "content": msg.content} for msg in req.history]
            if req.history else []
        )

        # Case 1: InBody scan image uploaded — restored regression.
        # Vision analysis isn't token-streamable, so we generate the full
        # reply then chunk it through to_stream_generator (already defined
        # above, was just orphaned).
        if req.inbody_image and req.inbody_image.strip():
            img_data = req.inbody_image.strip()
            if "," in img_data:
                img_data = img_data.split(",", 1)[1]

            temp_file = tempfile.NamedTemporaryFile(delete=False, suffix=".png")
            temp_file.write(base64.b64decode(img_data))
            temp_file.close()
            temp_path = temp_file.name

            def stream_inbody_and_cleanup(path: str, goal: str):
                try:
                    reply = process_inbody_and_consult(file_path=path, user_goal=goal)
                    for chunk in to_stream_generator(reply):
                        yield chunk
                except Exception as e:
                    yield f"\n[InBody analysis error: {str(e)}]"
                finally:
                    if os.path.exists(path):
                        os.remove(path)

            return StreamingResponse(
                stream_inbody_and_cleanup(temp_path, req.query),
                media_type="text/plain",
                headers=STREAM_HEADERS,
            )

      # Case 2: Nutrition label scan image uploaded
        if req.label_image and req.label_image.strip():
            img_data = req.label_image.strip()
            if "," in img_data:
                img_data = img_data.split(",", 1)[1]
        
            missing_padding = len(img_data) % 4
            if missing_padding:
                img_data += "=" * (4 - missing_padding)
        
            try:
                decoded_bytes = base64.b64decode(img_data)
            except Exception as b64_err:
                raise HTTPException(status_code=400, detail="Invalid Base64 image payload")
        
            temp_file = tempfile.NamedTemporaryFile(delete=False, suffix=".png")
            temp_file.write(decoded_bytes)
            temp_file.close()
            temp_path = temp_file.name
        
            def stream_label_and_cleanup(path: str, user_query: str):
                try:
                    yield "### 📷 Nutrition Label Analysis\n\n"
                    
                    # Step 1: Use Gemini Vision to extract the text & macros from the image
                    extracted_label_text = parse_nutrition_label(path)
                    
                    # Step 2: Construct a prompt combining the extracted macros with the user's request
                    combined_prompt = f"""The user uploaded a nutrition label image. Here is the extracted nutrition facts data:
        
        {extracted_label_text}
        
        User Request: {user_query or 'Analyze these macros and give me dietary advice.'}"""
        
                    # Step 3: Stream the response through your local FitPulse AI model
                    label_stream = stream_fitpulse_with_memory(
                        prompt=combined_prompt,
                        history=formatted_history,
                        use_rag=req.use_rag
                    )
                    
                    for chunk in to_stream_generator(label_stream):
                        yield chunk
        
                except Exception as e:
                    yield f"\n[Label analysis error: {str(e)}]"
                finally:
                    if os.path.exists(path):
                        os.remove(path)
        
            return StreamingResponse(
                stream_label_and_cleanup(temp_path, req.query),
                media_type="text/plain",
                headers=STREAM_HEADERS,
            )
        # Case 3: regular text chat — real token-by-token stream from the local model
        return StreamingResponse(
            stream_fitpulse_with_memory(
                prompt=req.query,        # fixed: was req.prompt (field doesn't exist on ChatRequest)
                history=formatted_history,
                use_rag=req.use_rag
            ),
            media_type="text/plain",
            headers=STREAM_HEADERS,
        )
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))

####################################################################

# **🟢 CELL 5: NGROK Tunnel**

In [ ]:
from pyngrok import ngrok
import uvicorn
import threading
import time
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()

# Set your Ngrok Auth Token
NGROK_TOKEN = user_secrets.get_secret("NGROK_TOKEN")
ngrok.set_auth_token(NGROK_TOKEN)


# 1. Close active pyngrok tunnel instances
try:
    for t in ngrok.get_tunnels():
        ngrok.disconnect(t.public_url)
    ngrok.kill()
    print("🧹 Cleared active pyngrok sessions.")
except Exception as e:
    print(f"Info: {e}")

# 2. Hard-kill any lingering ngrok background processes in Linux
os.system("pkill -9 -f ngrok")
time.sleep(1) # Small pause for socket cleanup
print("⚡ Killed lingering ngrok system processes.")


# Reset Ngrok tunnels
try:
    ngrok.kill()
except Exception:
    pass

# Open Ngrok tunnel on port 8000
public_url = ngrok.connect(8000, pooling_enabled=True )
print(f"\n🚀 REACT ENDPOINT URL:\n{public_url.public_url}/api/chat\n")

# Pass `app` directly as the Python object from Cell 6
config = uvicorn.Config(app, host="0.0.0.0", port=8000)
server = uvicorn.Server(config)

# Run Uvicorn in top-level await (or via background thread)
await server.serve()

🧹 Cleared active pyngrok sessions.
⚡ Killed lingering ngrok system processes.


INFO:     Started server process [468]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)



🚀 REACT ENDPOINT URL:
https://gumdrop-paralyses-replica.ngrok-free.dev/api/chat

🚀 FitPulse API starting up...
INFO:     154.180.239.57:0 - "POST /api/chat HTTP/1.1" 200 OK
INFO:     154.180.239.57:0 - "POST /api/chat HTTP/1.1" 200 OK
INFO:     154.180.239.57:0 - "OPTIONS /api/chat HTTP/1.1" 200 OK
INFO:     154.180.239.57:0 - "POST /api/chat HTTP/1.1" 200 OK
INFO:     154.180.239.57:0 - "POST /api/chat HTTP/1.1" 200 OK
INFO:     154.180.239.57:0 - "POST /api/chat HTTP/1.1" 200 OK
INFO:     154.180.239.57:0 - "OPTIONS /api/chat HTTP/1.1" 200 OK
INFO:     154.180.239.57:0 - "POST /api/chat HTTP/1.1" 200 OK
INFO:     154.180.239.57:0 - "POST /api/chat HTTP/1.1" 200 OK
INFO:     154.180.239.57:0 - "OPTIONS /api/chat HTTP/1.1" 200 OK
INFO:     154.180.239.57:0 - "POST /api/chat HTTP/1.1" 200 OK
